# KV260 — Phương án A: IP `ecg_f2a_top` + hiệu chỉnh theo bệnh nhân trên ARM
**File cần chép vào cùng thư mục trên board:** `ecg_f2a.bit`, `ecg_f2a.hwh`, `xecg_f2a_top_hw.h`, `lib_ecg_f2a.py`, `pa_adapt.py`,
`weights_f2a.h`, `test_beats_q.txt`, `test_rr_q.txt`, `test_logits_sim.txt`, `test_emb_sim.txt`, `test_head_adapted.txt`,
`test_logits_adapted.txt`, `f2a_board_data.npz`.

1. Kiểm chứng bit-exact với C-sim (T1: trọng số gốc, T3: trọng số hiệu chỉnh) — mức 5 của luồng kiểm chứng.
2. Đo latency một nhịp.
3. Chạy **toàn bộ DS2**: mỗi bản ghi thu vector đặc trưng 5 phút đầu → ARM hiệu chỉnh (`offset_int`, `gradient_anchor`) → ghi lớp đầu ra → phân loại phần còn lại; đo thời gian hiệu chỉnh trên ARM.

In [ ]:
import time, json, numpy as np
from pynq import Overlay
import lib_ecg_f2a as L, pa_adapt as pa
ol = Overlay('ecg_f2a.bit')
from pynq import Clocks
print('fclk0 truoc:', Clocks.fclk0_mhz, 'MHz')
Clocks.fclk0_mhz = 100          # thiet ke dong timing o 100 MHz (WNS +2.563 ns); khong chay 150 MHz
print('fclk0 sau  :', Clocks.fclk0_mhz, 'MHz')
regs = L.parse_hw_header('xecg_f2a_top_hw.h'); print({k: hex(v) for k, v in regs.items() if k in L.REQUIRED})
ip = L.ECGF2A(ol.ecg_f2a_top_0.mmio, regs)
W_G, B_G = L.parse_global_head('weights_f2a.h')        # lop dau ra toan cuc (seed 3, bias da gop offset)
ip.write_head(W_G, B_G)
w_rb, b_rb = ip.read_head(); print('Doc lai head_w/head_b khop:', (w_rb == W_G).all() and (b_rb == B_G).all())

## 1. Kiểm chứng bit-exact với C-sim

In [ ]:
B = np.loadtxt('test_beats_q.txt', dtype=np.int64); R = np.loadtxt('test_rr_q.txt', dtype=np.int64)
ip.write_head(W_G, B_G)
lg, em, lab = ip.infer_batch(B, R)
print('[T1] logit khop C-sim:', (lg == np.loadtxt('test_logits_sim.txt', dtype=np.int64)).all(),
      '| emb khop C-sim:', (em == np.loadtxt('test_emb_sim.txt', dtype=np.int64)).all())
H = np.loadtxt('test_head_adapted.txt', dtype=np.int64); ip.write_head(H[:128], H[128:])
lg3, _, _ = ip.infer_batch(B, R)
print('[T3] logit (trong so hieu chinh) khop C-sim:', (lg3 == np.loadtxt('test_logits_adapted.txt', dtype=np.int64)).all())
ip.write_head(W_G, B_G)

## 2. Latency một nhịp (AXI-inclusive: ghi đầu vào + chạy + đọc logit, emb, nhãn)

In [ ]:
for _ in range(20): ip.infer(B[0], R[0])
t0 = time.perf_counter()
for k in range(1000): ip.infer(B[k], R[k])
print(f'Latency trung binh: {(time.perf_counter() - t0) / 1000 * 1e3:.3f} ms/nhip (ECG_F2: 0.955 ms)')

## 3. Hiệu chỉnh theo bệnh nhân trên toàn bộ DS2 (5 phút đầu → hiệu chỉnh, phần còn lại → đánh giá)

In [ ]:
D = np.load('f2a_board_data.npz'); hp = json.loads(str(D['best_hp']))
beats, rrs, y, rec, t = D['beats_q'], D['rr_q'], D['y'], D['rec'], D['t']
W0 = W_G.reshape(4, 32).T / 128.0; b0 = B_G / 16384.0      # lop dau ra toan cuc (so thuc) cho pa_adapt
# vector dac trung cua nhip neo (DS1) lay TRUC TIEP tu IP -> bit-exact voi phan cung
ip.write_head(W_G, B_G)
_, EA, _ = ip.infer_batch(D['anc_beats_q'], D['anc_rr_q'])
pa.CURRENT_ANCHORS = (EA / 16.0, D['anc_y'])
cal = pa.split_first(t, rec, 5); ev = ~cal
print('Nhip hieu chinh:', cal.sum(), '| nhip danh gia:', ev.sum(), '| sieu tham so (chon tren DS1):', hp)

In [ ]:
t_all = time.perf_counter()
_, E_all, P_base = ip.infer_batch(beats, rrs)          # 1 luot voi trong so toan cuc: emb + nhan goc
print(f'Luot goc: {time.perf_counter() - t_all:.1f} s cho {len(y)} nhip')
PRED = {'none': P_base.copy()}; T_ADAPT = {}
for mth in ['offset_int', 'gradient_anchor']:
    PRED[mth] = P_base.copy(); T_ADAPT[mth] = []
    for r in np.unique(rec):
        m = rec == r; mc, me = m & cal, m & ev
        ta = time.perf_counter()
        W, b = pa.METHODS[mth](E_all[mc] / 16.0, y[mc], W0, b0, **hp[mth])      # tinh tren ARM
        w_i8, b_q14 = pa.head_to_registers(W, b)
        T_ADAPT[mth].append(time.perf_counter() - ta)
        ip.write_head(w_i8, b_q14)
        idx = np.where(me)[0]
        _, _, PRED[mth][idx] = ip.infer_batch(beats[idx], rrs[idx])   # phan loai lai tren IP voi lop dau ra moi
    ip.write_head(W_G, B_G)
    print(f'{mth}: thoi gian hieu chinh ARM trung binh {np.mean(T_ADAPT[mth]) * 1e3:.1f} ms/ban ghi (max {np.max(T_ADAPT[mth]) * 1e3:.1f} ms)')
for mth, P in PRED.items():
    print(f'{mth:16s}', pa.fmt(pa.metrics(y[ev], P[ev])))
np.savez('f2a_board_results.npz', **{f'pred_{k}': v for k, v in PRED.items()},
         **{f'tadapt_{k}': np.array(v) for k, v in T_ADAPT.items()})
print('Da luu f2a_board_results.npz')